# El precio es correcto

Hoy construimos una solución más compleja para estimar los precios de los bienes.

1. Este cuaderno: crea una base de datos RAG con nuestros 400 000 datos de entrenamiento
2. Cuaderno del día 2.1: visualiza en 2D
3. Cuaderno del día 2.2: visualiza en 3D
4. Cuaderno del día 2.3: crea y prueba una canalización RAG con GPT-4o-mini
5. Cuaderno del día 2.4: (a) recupera nuestro tasador de Bosque aleatorio (b) crea un tasador de conjunto que permita contribuciones de todos los tasadores

¡Uf! ¡Eso es mucho para hacer en un día!

## TEN EN CUENTA:

Ya tenemos un estimador de productos muy poderoso con nuestro LLM patentado y perfeccionado. ¡La mayoría de las personas estarían muy satisfechas con eso! La razón principal por la que agregamos estos pasos adicionales es para profundizar su experiencia con RAG y con los flujos de trabajo de Agentic.

In [1]:
# imports

import os
import re
import math
import json
from tqdm import tqdm
import random
from dotenv import load_dotenv
from huggingface_hub import login
import numpy as np
import pickle
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
import chromadb
from items import Item
from sklearn.manifold import TSNE
import plotly.graph_objects as go

/Users/isaac/workspace/llm_engineering/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# environment

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')
os.environ['HF_TOKEN'] = os.getenv('HF_TOKEN', 'your-key-if-not-using-env')
DB = "products_vectorstore"

In [3]:
# Log in en HuggingFace

hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## Volvamos a los archivos pkl

Aunque disfrutamos mucho de la curación de datos en la semana 6, probablemente no queramos pasar por todo ese proceso nuevamente.

Reutilicemos los archivos pkl que creamos entonces. Copie los archivos `train.pkl` y `test.pkl` de la carpeta de la semana 6 a esta carpeta de la semana 8, o también puede descargarlos desde aquí:

https://drive.google.com/drive/folders/1f_IZGybvs9o0J5sb3xmtTEQB3BXllzrW?usp=drive_link

In [4]:
# Con train.pkl en esta carpeta, puedes ejecutar:

with open('train.pkl', 'rb') as file:
    train = pickle.load(file)

In [5]:
train[0].prompt

'How much does this cost to the nearest dollar?\n\nDelphi FG0166 Fuel Pump Module\nDelphi brings 80 years of OE Heritage into each Delphi pump, ensuring quality and fitment for each Delphi part. Part is validated, tested and matched to the right vehicle application Delphi brings 80 years of OE Heritage into each Delphi assembly, ensuring quality and fitment for each Delphi part Always be sure to check and clean fuel tank to avoid unnecessary returns Rigorous OE-testing ensures the pump can withstand extreme temperatures Brand Delphi, Fit Type Vehicle Specific Fit, Dimensions LxWxH 19.7 x 7.7 x 5.1 inches, Weight 2.2 Pounds, Auto Part Position Unknown, Operation Mode Mechanical, Manufacturer Delphi, Model FUEL PUMP, Dimensions 19.7\n\nPrice is $227.00'

# Ahora crea un almacén de datos de Chroma

En la semana 5, creamos un almacén de datos de Chroma con 123 documentos que representan fragmentos de objetos de nuestra empresa ficticia Insurellm.

¡Ahora crearemos un almacén de datos de Chroma con 400 000 productos de nuestro conjunto de datos de entrenamiento! ¡Se está volviendo real!

Ten en cuenta que no usaremos LangChain, pero la API es muy sencilla y coherente con la anterior.

In [6]:
client = chromadb.PersistentClient(path=DB)

In [7]:
# Comprueba si la colección existe y elimínala si es así
collection_name = "products"
existing_collection_names = [collection.name for collection in client.list_collections()]
if collection_name in existing_collection_names:
    client.delete_collection(collection_name)
    print(f"Deleted existing collection: {collection_name}")

collection = client.create_collection(collection_name)

Deleted existing collection: products


# Presentamos SentenceTransformer

El modelo all-MiniLM es un modelo muy útil de HuggingFace que asigna oraciones y párrafos a un espacio vectorial denso de 384 dimensiones y es ideal para tareas como la búsqueda semántica.

https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2

Puede ejecutarse localmente con bastante rapidez.

La última vez usamos incrustaciones de OpenAI para producir incrustaciones vectoriales. Beneficios en comparación con las incrustaciones de OpenAI:
1. ¡Es gratis y rápido!
2. Podemos ejecutarlo localmente, por lo que los datos nunca salen de nuestra caja; puede ser útil si estás creando un RAG personal

In [8]:
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2023.93it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
# Pasa una lista de textos y obtén una matriz numpy de vectores

vector = model.encode(["Hola, ¿cómo estás?"])[0]

In [10]:
vector

array([-1.15368993e-03,  5.10658771e-02,  1.53511474e-02,  2.32141148e-02,
       -5.79879768e-02, -7.84520409e-04,  1.17281653e-01, -1.06943157e-02,
        2.04955917e-02, -1.72842871e-02,  6.00087419e-02, -9.82523337e-03,
       -8.24677274e-02,  2.87922919e-02,  1.30775616e-01,  4.28776406e-02,
       -2.61534639e-02, -2.76041664e-02,  2.44764276e-02, -7.89872371e-03,
        6.07455038e-02,  6.74455287e-03, -7.15552717e-02,  1.07552804e-01,
       -6.41963854e-02, -3.59639861e-02,  4.00140435e-02,  1.86638292e-02,
       -3.14408652e-02, -7.57469609e-02, -3.55393179e-02,  4.59692553e-02,
        2.67851371e-02,  2.39503253e-02,  1.50616104e-02, -1.37406457e-02,
        7.68323988e-02, -1.17426485e-01, -4.72090505e-02,  4.06149849e-02,
       -1.25587583e-01, -1.39257796e-02,  5.64728212e-03,  3.93794104e-03,
        1.87711278e-03, -1.02197789e-01, -4.06030472e-03,  5.26404530e-02,
        5.41949980e-02, -2.73344629e-02, -1.07816510e-01, -9.07929335e-03,
       -6.23978376e-02,  

In [11]:
def description(item):
    text = item.prompt.replace("How much does this cost to the nearest dollar?\n\n", "")
    return text.split("\n\nPrice is $")[0]

In [12]:
description(train[0])

'Delphi FG0166 Fuel Pump Module\nDelphi brings 80 years of OE Heritage into each Delphi pump, ensuring quality and fitment for each Delphi part. Part is validated, tested and matched to the right vehicle application Delphi brings 80 years of OE Heritage into each Delphi assembly, ensuring quality and fitment for each Delphi part Always be sure to check and clean fuel tank to avoid unnecessary returns Rigorous OE-testing ensures the pump can withstand extreme temperatures Brand Delphi, Fit Type Vehicle Specific Fit, Dimensions LxWxH 19.7 x 7.7 x 5.1 inches, Weight 2.2 Pounds, Auto Part Position Unknown, Operation Mode Mechanical, Manufacturer Delphi, Model FUEL PUMP, Dimensions 19.7'

In [13]:
for i in tqdm(range(0, len(train), 1000)):
    documents = [description(item) for item in train[i: i+1000]]
    vectors = model.encode(documents).astype(float).tolist()
    metadatas = [{"category": item.category, "price": item.price} for item in train[i: i+1000]]
    ids = [f"doc_{j}" for j in range(i, i+1000)]
    collection.add(
        ids=ids,
        documents=documents,
        embeddings=vectors,
        metadatas=metadatas
    )

100%|██████████| 400/400 [41:37<00:00,  6.24s/it]
